In [ ]:
from statemachine import State, StateMachine
import simpy

class TrafficLight(StateMachine):
  red = State('Red', value=1, initial=True)
  green = State('Green', value=2)
  yellow = State('Yellow', value=3)
  cycle = green.to(yellow) | yellow.to(red) | red.to(green)

  def __init__(self, env, name):
    StateMachine.__init__(self)
    self.env = env
    self.name = name
    self.env.process(self.on_enter_red())

  def on_enter_red(self):
    yield self.env.timeout(30)
    print(f"{self.name} done RED: {self.env.now}")
    self.cycle()

  def on_enter_green(self):
    yield self.env.timeout(25)
    print(f"{self.name} done GREEN: {self.env.now}")
    self.cycle()
  
  def on_enter_yellow(self):
    yield self.env.timeout(5)
    print(f"{self.name} done YELLOW: {self.env.now}")
    self.cycle()

  def after_cycle(self):
    if str(self.current_state) == 'Red':
      self.env.process(self.on_enter_red())
    elif str(self.current_state) == 'Green':
      self.env.process(self.on_enter_green())
    elif str(self.current_state) == 'Yellow':
      self.env.process(self.on_enter_yellow())

def main():
  env = simpy.Environment()
  t1 = TrafficLight(env, "Water St.")
  t2 = TrafficLight(env, "Main St.")
  env.run(until=200)

if __name__ == '__main__':
  main()

Water St. done RED: 30
Main St. done RED: 30
Water St. done GREEN: 55
Main St. done GREEN: 55
Water St. done YELLOW: 60
Main St. done YELLOW: 60
Water St. done RED: 90
Main St. done RED: 90
Water St. done GREEN: 115
Main St. done GREEN: 115
Water St. done YELLOW: 120
Main St. done YELLOW: 120
Water St. done RED: 150
Main St. done RED: 150
Water St. done GREEN: 175
Main St. done GREEN: 175
Water St. done YELLOW: 180
Main St. done YELLOW: 180


In [74]:
def simple_coroutine():
    print("starting coroutine")
    x = yield  # Receive an initial value
    print(f"Received: {x}")
    y = yield x + 5  # Send x + 1, then receive another value
    print(f"Received: {y}")
    z = yield y + 1  # Send y + 1, then receive another value

co = simple_coroutine() 
next(co)  # Prime the coroutine (send None implicitly)

print(co.send(1))  # Send 1 to the coroutine
print(co.send(2))  # Send 2 to the coroutine

starting coroutine
Received: 1
6
Received: 2
3
